# Running this pipeline with real XGBoost

My own sandboxed environment can't install `xgboost`, so `04_scale_and_select.py` only ever runs there with the scikit-learn fallback. Colab already has the real thing, so this notebook just reruns the pipeline here instead.

Run the cells in order (Runtime > Run all). When asked, upload the three synthetic CSVs from `synthetic_data_repo/data/` on your computer:
- `synthetic_production.csv`
- `synthetic_weather_FW_Funchal.csv`
- `synthetic_weather_LPMA.csv`

At the end, `processed_results.zip` downloads automatically with the final per-target datasets and the real-XGBoost importance tables, so those can be committed into the repo in place of the current fallback versions.


## 1. Check that real XGBoost is available


In [ ]:
import sys
try:
    import xgboost
    print('xgboost', xgboost.__version__, '- the real thing, not the sklearn fallback.')
except ImportError:
    print('xgboost missing, installing...')
    !pip install -q xgboost
    import xgboost
    print('xgboost', xgboost.__version__, 'installed.')


## 2. Upload the three synthetic data CSVs


In [ ]:
import os
from google.colab import files

os.makedirs('synthetic_data_repo/data', exist_ok=True)
os.makedirs('synthetic_data_repo/pipeline', exist_ok=True)

print('Select all 3 files at once in the upload dialog:')
print(' - synthetic_production.csv')
print(' - synthetic_weather_FW_Funchal.csv')
print(' - synthetic_weather_LPMA.csv')
uploaded = files.upload()

expected = {
    'synthetic_production.csv',
    'synthetic_weather_FW_Funchal.csv',
    'synthetic_weather_LPMA.csv',
}
missing = expected - set(uploaded.keys())
if missing:
    raise RuntimeError(f'Missing expected file(s): {missing}. Re-run this cell and upload all three.')

for fname in uploaded:
    os.replace(fname, f'synthetic_data_repo/data/{fname}')
print('All 3 files placed in synthetic_data_repo/data/')


## 3. Write the pipeline scripts

These are byte-identical to the versions in your `synthetic_data_repo/pipeline/` folder. Only `04_scale_and_select.py` actually behaves differently here, since this environment has xgboost installed.


In [ ]:
%%writefile synthetic_data_repo/pipeline/01_merge_data.py
"""
01_merge_data.py

Step 1 of the pipeline: lines up two weather stations and the production
meter readings on a common hourly clock. Nothing clever yet, but getting
this join right is what every later feature and model depends on.

Maps to notebook sections 1.1, 1.2.1 and 1.2.2 of
`COPY -Final code/preprocessing/data_preprocessing_and_modeling.ipynb`
(roughly cells 6-72), adapted to run on the synthetic CSVs instead of
Production.pkl / FW_Funchal.pkl / LPMA.pkl. Same order as the notebook:

1. Load the two synthetic hourly weather files and the synthetic 15-minute
   production file.
2. Weather merge (1.2.1): convert `winddir` (degrees) to sine/cosine on
   both stations, since you can't take a circular mean on raw degrees, drop
   the original degree/radian columns, then average the two stations'
   remaining numeric columns with a plain arithmetic mean. One quirk kept
   on purpose: the notebook never renormalises the sin/cos pair back onto
   the unit circle after averaging, so neither does this.
3. Production/weather merge (1.2.2): drop duplicate timestamps on both
   sides, keeping the first record. Production values are a rate ("if
   sustained for an hour"), so the energy columns get multiplied by 0.25
   before resampling 15-min data up to hourly bins (sum for flow/energy
   columns, first for metadata). That resample step quietly creates a
   handful of extra hourly rows with NaN metadata whenever a whole hour has
   no underlying 15-min rows, most notably the missing hour at a
   spring-forward DST transition, and those get dropped. It also means an
   all-NaN hour sums to 0.0 instead of NaN for the `sum`-aggregated columns
   (this bites Hydropower during its real missingness window, November
   2021). Left as-is here, same as the notebook, because the imputer step
   in 02_preprocess.py already treats `isna() | eq(0)` as "missing" for
   Hydropower, so it gets handled downstream rather than patched twice.
   Finally, the two frames get aligned on timestamps that exist in both and
   concatenated column-wise.

One real adaptation versus the notebook, worth flagging: the real data had
exactly one DST mismatch (26 March in weather vs. 28 March in production,
2023 only), so the notebook just hard-codes dropping those two specific
timestamps. The synthetic generator reproduces a spring-forward gap in
production every year but has no DST gap in weather at all, so a single
hard-coded date pair wouldn't cover it. This script generalises the intent
instead: drop whichever hour exists on one side but not the other around
each DST transition, computed as a symmetric difference between the two
indices rather than listed by hand. Same effect, same row count, if the
synthetic data happened to match the real pattern exactly. It also happens
to clean up one more mismatch for free: the synthetic production series
runs one hour later on its last day than the synthetic weather series, an
artefact of how the two generators pick their end boundary, not a DST
issue, and that trailing hour gets caught by the same step.

Output: `data/merged_hourly.csv`, ready for `02_preprocess.py`.
"""

from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path(__file__).resolve().parent.parent / "data"
PRODUCTION_CSV = DATA_DIR / "synthetic_production.csv"
WEATHER_FW_CSV = DATA_DIR / "synthetic_weather_FW_Funchal.csv"
WEATHER_LPMA_CSV = DATA_DIR / "synthetic_weather_LPMA.csv"
OUT_PATH = DATA_DIR / "merged_hourly.csv"

PRODUCTION_FLOW_COLS = ["Hydropower", "Fossil", "Natural Gas", "Biomass", "Wind", "Solar"]

PRODUCTION_HOURLY_AGG = {
    "Date": "first",
    "Hydropower": "sum",
    "Fossil": "sum",
    "Natural Gas": "sum",
    "Biomass": "sum",
    "Wind": "sum",
    "Solar": "sum",
    "Curtailed Wind Energy": "sum",
    "Year": "first",
    "Wind Energy Capacity": "first",
    "Hydropower Energy Capacity": "first",
    "Solar Energy Capacity": "first",
    "Biomass Energy Capacity": "first",
    "Fossil Energy Capacity": "first",
    "Natural Gas Energy Capacity": "first",
    "Month Name": "first",
    "Day of Month": "first",
    "Month": "first",
    "Season": "first",
    "Hour of the day": "first",
    "Hour of the Year": "first",
}


def load_raw():
    production = pd.read_csv(PRODUCTION_CSV, parse_dates=["Hora"]).set_index("Hora")
    fw_funchal = pd.read_csv(WEATHER_FW_CSV, parse_dates=["date_time"]).set_index("date_time")
    lpma = pd.read_csv(WEATHER_LPMA_CSV, parse_dates=["date_time"]).set_index("date_time")
    return production, fw_funchal, lpma


def merge_weather(fw_funchal: pd.DataFrame, lpma: pd.DataFrame) -> pd.DataFrame:
    """Notebook 1.2.1 - circular handling of winddir, then plain mean of stations."""
    fw_funchal = fw_funchal.copy()
    lpma = lpma.copy()

    for station in (fw_funchal, lpma):
        station["winddir_rad"] = np.deg2rad(station["winddir"])
        station["winddir_sin"] = np.sin(station["winddir_rad"])
        station["winddir_cos"] = np.cos(station["winddir_rad"])
        station.drop(columns=["winddir", "winddir_rad"], inplace=True)

    weather_data = (fw_funchal + lpma) / 2
    return weather_data


def merge_production_weather(production: pd.DataFrame, weather_data: pd.DataFrame):
    """Notebook 1.2.2 - dedup, rate->mean, hourly resample, DST alignment, concat."""
    stats = {}

    stats["production_raw_rows"] = len(production)
    stats["weather_raw_rows"] = len(weather_data)

    # (a) duplicate handling - keep first, both datasets
    n_dup_production = int(production.index.duplicated().sum())
    n_dup_weather = int(weather_data.index.duplicated().sum())
    production = production[~production.index.duplicated(keep="first")]
    weather_data = weather_data[~weather_data.index.duplicated(keep="first")]
    stats["production_duplicates_dropped"] = n_dup_production
    stats["weather_duplicates_dropped"] = n_dup_weather

    # (b) rate -> "energy if sustained for an hour" for the 15-min slices,
    # then resample 15-min -> hourly
    production = production.copy()
    production.loc[:, PRODUCTION_FLOW_COLS] = production[PRODUCTION_FLOW_COLS] * 0.25
    production_hourly = production.resample("h").agg(PRODUCTION_HOURLY_AGG)
    stats["production_hourly_rows_before_dst_fix"] = len(production_hourly)

    # (c) resample-created NaN-metadata rows (the "4 extra rows" quirk)
    nan_date_rows = production_hourly.index[production_hourly["Date"].isna()]
    production_hourly = production_hourly.dropna(subset=["Date"])
    stats["dropped_nan_metadata_rows"] = len(nan_date_rows)
    stats["dropped_nan_metadata_timestamps"] = [str(t) for t in nan_date_rows]

    # (d) DST / boundary alignment - generalised symmetric-difference fix
    # (see module docstring for why this differs from the notebook's
    # single hard-coded date pair)
    weather_only = weather_data.index.difference(production_hourly.index)
    production_only = production_hourly.index.difference(weather_data.index)
    weather_aligned = weather_data.drop(index=weather_only, errors="ignore")
    production_aligned = production_hourly.drop(index=production_only, errors="ignore")
    stats["weather_only_timestamps_dropped"] = [str(t) for t in weather_only]
    stats["production_only_timestamps_dropped"] = [str(t) for t in production_only]

    remaining_diff = production_aligned.index.symmetric_difference(weather_aligned.index)
    assert len(remaining_diff) == 0, f"Indices still differ: {remaining_diff}"

    production_aligned = production_aligned.sort_index()
    weather_aligned = weather_aligned.sort_index()

    # (e) merge
    merged = pd.concat([production_aligned, weather_aligned], axis=1)
    stats["merged_rows"] = len(merged)
    stats["merged_cols"] = merged.shape[1]

    return merged, stats


def main():
    production, fw_funchal, lpma = load_raw()
    weather_data = merge_weather(fw_funchal, lpma)
    merged, stats = merge_production_weather(production, weather_data)

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    out_path = OUT_PATH
    merged.to_csv(out_path)

    print("=== 01_merge_data.py ===")
    for k, v in stats.items():
        print(f"{k}: {v}")
    print(f"Wrote merged hourly dataframe -> {out_path} shape={merged.shape}")
    return merged, stats


if __name__ == "__main__":
    main()


In [ ]:
%%writefile synthetic_data_repo/pipeline/02_preprocess.py
"""
02_preprocess.py

Step 2: cleans up the inevitable mess in two years of real-world sensor and
meter data, missing hours, DST jumps, gaps that need imputing, so the
models downstream are learning signal, not sensor noise.

Maps to notebook sections 2.1, 2.2, the second 2.2 (the notebook reuses
that section number for the cyclical encoding step, kept as-is here), and
2.3, cells roughly 83-125. Same order as the notebook:

Section 2.1 casts all float64 columns to float32 (Season stays as-is), and
drops the columns the original analysis found didn't add anything: Month
Name, Hour of the Year, Year, Curtailed Wind Energy, snowdepth, Hour of the
day.

Section 2.2 splits on fixed dates into a 70/15/15 train/val/test split on
the hourly merged frame (train through 2023-07-27 06:00, val through
2024-02-13 02:00, test after that), then the second "2.2" step maps Season
to an integer code, adds day_of_week and is_weekend, and replaces Day of
Month, Month, Season and day_of_week with their sine/cosine pairs. Each
split gets its own cyclical encoding fit independently, though since these
are just deterministic trig transforms rather than anything learned from
the data, that doesn't actually matter for correctness.

Section 2.3 imputes two different ways, train split only in both cases:
linear interpolation for visibility, solarradiation, solarenergy and
uvindex (barely any missing values to begin with), and a Random Forest
imputer for the two variables with real gaps. Hydropower gets trained on
temp, dew, Season_cos, Month_sin and Month_cos, excluding November 2021
from training since that's the exact window being imputed, then applied to
any November 2021 row where Hydropower is NaN or exactly 0 (that 0 case is
the resample-sum artefact described in 01_merge_data.py). windgust gets
trained on windspeed and Wind, using every row with a real windgust value,
applied to whatever's left missing.

Worth noting: the original only imputes the train split, val and test are
left alone at this stage. Kept the same way here. That was presumably fine
on the real data because both missingness windows fall inside train, and
this script prints whatever NaNs are left over in val/test so that
assumption can actually be checked against the synthetic data rather than
just assumed to hold.

Input:  data/merged_hourly.csv (from 01_merge_data.py)
Output: data/train_hourly.csv, data/val_hourly.csv, data/test_hourly.csv
"""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

DATA_DIR = Path(__file__).resolve().parent.parent / "data"
IN_PATH = DATA_DIR / "merged_hourly.csv"

TRAIN_END = "2023-07-27 06:00"
VAL_START = "2023-07-27 07:00"
VAL_END = "2024-02-13 02:00"
TEST_START = "2024-02-13 03:00"

COLUMNS_TO_DROP = [
    "Month Name",
    "Hour of the Year",
    "Year",
    "Curtailed Wind Energy",
    "snowdepth",
    "Hour of the day",
]

SEASON_ENCODING = {"Winter": 0, "Spring": 1, "Summer": 2, "Autumn": 3}

INTERPOLATE_COLS = ["visibility", "solarradiation", "solarenergy", "uvindex"]

HYDROPOWER_RF_FEATURES = ["temp", "dew", "Season_cos", "Month_sin", "Month_cos"]
WINDGUST_RF_FEATURES = ["windspeed", "Wind"]


def load_merged() -> pd.DataFrame:
    df = pd.read_csv(IN_PATH, index_col=0, parse_dates=True)
    return df


def initial_adjustments(df: pd.DataFrame) -> pd.DataFrame:
    """Notebook 2.1 - datatypes + dropping unused columns."""
    df = df.copy()
    exclude = ["Season", "Month Name"]
    float_cols = df.select_dtypes(include=["float64"]).columns.difference(exclude)
    df[float_cols] = df[float_cols].astype("float32")

    for col in COLUMNS_TO_DROP:
        if col in df.columns:
            del df[col]
    return df


def train_val_test_split(df: pd.DataFrame):
    """Notebook 2.2 - fixed-date 70/15/15 split."""
    train = df.loc[:TRAIN_END].copy()
    val = df.loc[VAL_START:VAL_END].copy()
    test = df.loc[TEST_START:].copy()

    for part in (train, val, test):
        for col in ("Day of Month", "Month"):
            part[col] = part[col].astype(int)

    return train, val, test


def encode_cyclical_feature(parts, col, max_val):
    for df in parts:
        df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / max_val).astype("float32")
        df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / max_val).astype("float32")
        df.drop(columns=[col], inplace=True)


def data_transformation(train, val, test):
    """Notebook 2.2 (duplicate heading) - Season encoding, day_of_week/is_weekend,
    cyclical sin/cos encoding."""
    parts = [train, val, test]

    for df in parts:
        df["Season"] = df["Season"].map(SEASON_ENCODING).astype("int32")

    for df in parts:
        df["day_of_week"] = df.index.dayofweek

    for df in parts:
        df["is_weekend"] = (df["day_of_week"] >= 5).astype("int32")

    encode_cyclical_feature(parts, "Day of Month", 31)
    encode_cyclical_feature(parts, "Month", 12)
    encode_cyclical_feature(parts, "Season", 4)
    encode_cyclical_feature(parts, "day_of_week", 7)

    return train, val, test


def linear_interpolate_train(train: pd.DataFrame) -> pd.DataFrame:
    """Notebook 2.3.1."""
    train = train.copy()
    train[INTERPOLATE_COLS] = train[INTERPOLATE_COLS].interpolate(
        method="linear", limit_direction="both"
    )
    return train


def rf_impute_hydropower(train: pd.DataFrame) -> pd.DataFrame:
    """Notebook 2.3.2 - Hydropower RF imputer, trained excluding Nov 2021,
    applied to Nov 2021 rows that are NaN or exactly 0."""
    train = train.copy()
    target = "Hydropower"
    features = HYDROPOWER_RF_FEATURES

    df_train = train[train[target].notna()].copy()
    mask_nov_2021 = (df_train.index.month == 11) & (df_train.index.year == 2021)
    X_train = df_train.loc[~mask_nov_2021, features]
    y_train = df_train.loc[~mask_nov_2021, target]

    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    mask_missing_nov2021 = (
        (train.index.month == 11)
        & (train.index.year == 2021)
        & (train[target].isna() | train[target].eq(0))
    )
    X_missing = train.loc[mask_missing_nov2021, features]

    n_imputed = 0
    if not X_missing.empty:
        y_pred = model.predict(X_missing)
        train.loc[mask_missing_nov2021, target] = y_pred.astype(np.float32)
        n_imputed = int(mask_missing_nov2021.sum())

    return train, n_imputed


def rf_impute_windgust(train: pd.DataFrame) -> pd.DataFrame:
    """Notebook 2.3.2 - windgust RF imputer, trained on all non-null rows,
    applied to all remaining NaN rows."""
    train = train.copy()
    features = WINDGUST_RF_FEATURES

    train_known = train[train["windgust"].notna()]
    X_train = train_known[features]
    y_train = train_known["windgust"]

    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)

    mask_missing = train["windgust"].isna()
    n_imputed = int(mask_missing.sum())
    if n_imputed:
        X_missing = train.loc[mask_missing, features]
        y_missing_pred = rf_model.predict(X_missing)
        train.loc[mask_missing, "windgust"] = y_missing_pred.astype(np.float32)

    return train, n_imputed


def main():
    stats = {}
    df = load_merged()
    stats["merged_rows"] = len(df)

    df = initial_adjustments(df)
    stats["after_initial_adjustments_cols"] = df.shape[1]

    train, val, test = train_val_test_split(df)
    stats["train_rows"] = len(train)
    stats["val_rows"] = len(val)
    stats["test_rows"] = len(test)

    train, val, test = data_transformation(train, val, test)
    stats["cols_after_cyclical_encoding"] = train.shape[1]

    train = linear_interpolate_train(train)
    stats["train_na_after_interpolation"] = {
        c: int(train[c].isna().sum()) for c in INTERPOLATE_COLS
    }

    train, n_hydro_imputed = rf_impute_hydropower(train)
    stats["hydropower_rf_imputed_rows"] = n_hydro_imputed
    stats["hydropower_remaining_na_train"] = int(train["Hydropower"].isna().sum())

    train, n_windgust_imputed = rf_impute_windgust(train)
    stats["windgust_rf_imputed_rows"] = n_windgust_imputed
    stats["windgust_remaining_na_train"] = int(train["windgust"].isna().sum())

    # Diagnostics: leftover NaNs in val/test (not imputed at this stage,
    # matching the original notebook's scope - see module docstring)
    stats["val_remaining_na_total"] = int(val.isna().sum().sum())
    stats["test_remaining_na_total"] = int(test.isna().sum().sum())
    stats["val_na_by_col"] = {c: int(v) for c, v in val.isna().sum().items() if v > 0}
    stats["test_na_by_col"] = {c: int(v) for c, v in test.isna().sum().items() if v > 0}

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    train.to_csv(DATA_DIR / "train_hourly.csv")
    val.to_csv(DATA_DIR / "val_hourly.csv")
    test.to_csv(DATA_DIR / "test_hourly.csv")

    print("=== 02_preprocess.py ===")
    for k, v in stats.items():
        print(f"{k}: {v}")

    return train, val, test, stats


if __name__ == "__main__":
    main()


In [ ]:
%%writefile synthetic_data_repo/pipeline/03_feature_engineering.py
"""
03_feature_engineering.py

Step 3: turns raw hourly readings into the lagged, rolling-window features
that actually let a model learn what usually happens next for a given
energy source, instead of just reacting to the latest timestamp. Reproduces
notebook sections 2.4 (daily aggregation) and 2.5 (lags and rolling means),
roughly cells 129-137.

Daily aggregation (2.4) resamples each hourly train/val/test frame to daily
frequency with a per-column aggregator, matching the notebook's
`variables_to_agg` dict column for column: the production targets
(Hydropower, Fossil, Natural Gas, Biomass, Wind, Solar) and precip sum over
the day, the continuous weather variables (temp, feelslike, humidity, dew,
windgust, windspeed, pressure, visibility, cloudcover, solarradiation,
solarenergy, winddir_sin, winddir_cos) average, and the capacity columns,
precipprob, uvindex, is_weekend, and the already-cyclical-encoded calendar
columns (Day of Month_sin/cos, Month_sin/cos, Season_sin/cos,
day_of_week_sin/cos) just take the first value of the day. The dict is
copied verbatim, including its duplicate 'dew' and 'is_weekend' keys; a
Python dict just keeps whichever assignment came last, which is harmless
here since both duplicates use the same aggregator anyway.

Feature engineering (2.5) adds 1/3/7-day lags and 3/7-day rolling means for
Hydropower, Wind, Solar, Biomass, humidity, solarradiation, windgust and
windspeed. The rolling means are taken over the prior day's window
(`shift(1)` first) so a day's own value never leaks into its own rolling
mean. Lags and rolling means are computed on train+val+test concatenated
together, so val's and test's first few lagged values can be filled in from
the tail of the split before them, then the three frames are split back
apart. The first 7 rows of train are dropped afterward, since they're left
with NaNs in the lag-7/roll-7 columns once train has nothing further back to
draw on.

Input:  data/train_hourly.csv, data/val_hourly.csv, data/test_hourly.csv
Output: data/train_daily.csv, data/val_daily.csv, data/test_daily.csv
"""

from pathlib import Path

import pandas as pd

DATA_DIR = Path(__file__).resolve().parent.parent / "data"

VARIABLES_TO_AGG = {
    "Hydropower": "sum",
    "Fossil": "sum",
    "Natural Gas": "sum",
    "Biomass": "sum",
    "Wind": "sum",
    "Solar": "sum",
    "Wind Energy Capacity": "first",
    "Hydropower Energy Capacity": "first",
    "Solar Energy Capacity": "first",
    "Biomass Energy Capacity": "first",
    "Fossil Energy Capacity": "first",
    "Natural Gas Energy Capacity": "first",
    "temp": "mean",
    "feelslike": "mean",
    "humidity": "mean",
    "dew": "mean",
    "precip": "sum",
    "precipprob": "first",
    "windgust": "mean",
    "windspeed": "mean",
    "pressure": "mean",
    "visibility": "mean",
    "cloudcover": "mean",
    "solarradiation": "mean",
    "solarenergy": "mean",
    "uvindex": "first",
    "is_weekend": "first",
    "winddir_sin": "mean",
    "winddir_cos": "mean",
    "Day of Month_sin": "first",
    "Day of Month_cos": "first",
    "Month_sin": "first",
    "Month_cos": "first",
    "Season_sin": "first",
    "Season_cos": "first",
    "day_of_week_sin": "first",
    "day_of_week_cos": "first",
}

LAG_ROLL_COLS = [
    "Hydropower",
    "Wind",
    "Solar",
    "Biomass",
    "humidity",
    "solarradiation",
    "windgust",
    "windspeed",
]
LAGS = [1, 3, 7]
ROLLS = [3, 7]


def load_hourly_splits():
    train = pd.read_csv(DATA_DIR / "train_hourly.csv", index_col=0, parse_dates=True)
    val = pd.read_csv(DATA_DIR / "val_hourly.csv", index_col=0, parse_dates=True)
    test = pd.read_csv(DATA_DIR / "test_hourly.csv", index_col=0, parse_dates=True)
    return train, val, test


def daily_aggregate(train, val, test):
    """Notebook 2.4."""
    cols = [c for c in VARIABLES_TO_AGG if c in train.columns]
    agg = {c: VARIABLES_TO_AGG[c] for c in cols}
    train_d = train.resample("D").agg(agg)
    val_d = val.resample("D").agg(agg)
    test_d = test.resample("D").agg(agg)
    return train_d, val_d, test_d


def create_lag_rolling_features(train, val, test, cols, lags=LAGS, rolls=ROLLS, full_window=True):
    """Notebook 2.5 - verbatim port of `create_lag_rolling_features`."""
    train = train.sort_index()
    val = val.sort_index()
    test = test.sort_index()

    combined = pd.concat([train, val, test], axis=0)
    for col in cols:
        for lag in lags:
            combined[f"{col}_lag{lag}"] = combined[col].shift(lag)
        for w in rolls:
            minp = w if full_window else 1
            combined[f"{col}_roll{w}"] = (
                combined[col].shift(1).rolling(window=w, min_periods=minp).mean()
            )

    train_out = combined.loc[train.index]
    val_out = combined.loc[val.index]
    test_out = combined.loc[test.index]
    return train_out, val_out, test_out


def main():
    stats = {}
    train, val, test = load_hourly_splits()
    stats["train_hourly_rows"] = len(train)
    stats["val_hourly_rows"] = len(val)
    stats["test_hourly_rows"] = len(test)

    train_d, val_d, test_d = daily_aggregate(train, val, test)
    stats["train_daily_rows"] = len(train_d)
    stats["val_daily_rows"] = len(val_d)
    stats["test_daily_rows"] = len(test_d)
    stats["daily_cols"] = train_d.shape[1]

    train_d, val_d, test_d = create_lag_rolling_features(train_d, val_d, test_d, LAG_ROLL_COLS)
    stats["cols_after_lag_roll"] = train_d.shape[1]

    rows_before_trim = len(train_d)
    train_d = train_d.iloc[7:]
    stats["train_rows_dropped_for_lag7_nans"] = rows_before_trim - len(train_d)
    stats["train_daily_rows_final"] = len(train_d)
    stats["val_daily_rows_final"] = len(val_d)
    stats["test_daily_rows_final"] = len(test_d)

    # Worth a flag: the boundary calendar date between two adjacent splits
    # shows up twice. Each split gets resampled to daily frequency on its
    # own (see daily_aggregate above), so the single day where train ends
    # and val begins, or val ends and test begins, has a partial day's
    # worth of hours on each side, and both partial-day rows end up
    # sharing the same date label after the lag/rolling merge. That's
    # inherited straight from the notebook's own combination of
    # create_lag_rolling_features and resample('D'), not something this
    # script introduces, but anything downstream that joins or merges on
    # these frames by index rather than by position needs to know about
    # it - see the note on _attach_target in 04_scale_and_select.py.
    stats["train_daily_duplicate_dates"] = [
        str(d) for d in train_d.index[train_d.index.duplicated(keep=False)].unique()
    ]
    stats["val_daily_duplicate_dates"] = [
        str(d) for d in val_d.index[val_d.index.duplicated(keep=False)].unique()
    ]
    stats["test_daily_duplicate_dates"] = [
        str(d) for d in test_d.index[test_d.index.duplicated(keep=False)].unique()
    ]

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    train_d.to_csv(DATA_DIR / "train_daily.csv")
    val_d.to_csv(DATA_DIR / "val_daily.csv")
    test_d.to_csv(DATA_DIR / "test_daily.csv")

    print("=== 03_feature_engineering.py ===")
    for k, v in stats.items():
        print(f"{k}: {v}")

    return train_d, val_d, test_d, stats


if __name__ == "__main__":
    main()


In [ ]:
%%writefile synthetic_data_repo/pipeline/04_scale_and_select.py
"""
04_scale_and_select.py

Step 4: scales everything onto comparable ranges and lets XGBoost vote on
which features earn a place in the final models, rather than including
every variable and hoping for the best. Reproduces notebook sections 2.6
(scaling) and 2.7 (feature selection), roughly cells 141-173.

Scaling (2.6) applies a sqrt transform first, to every numeric feature
column except the four targets (Solar, Wind, Hydropower, Biomass) and any
cyclical column ending in '_sin'/'_cos', then MinMax scaling on top. Two
rounds are built, matching the original: round 1 fits a MinMaxScaler on
train only and applies it to train/val/test, producing X_train_<target>,
X_val_<target>, X_test_<target> plus each target's own sqrt+MinMax y
(scaler fit on that target's train y only). Round 2 concatenates train and
val into `trainval`, refits the X scaler and each target's y scaler on
that, and applies them to trainval/test, producing X_trainval_<target>,
X_test_<target>_tv and the matching y_trainval/y_test_tv pairs. Every
target starts from the same full scaled X copy; features only get pared
down per target in 2.7. All of the scalers, X and each target's y, for both
rounds, are saved to `data/processed/scalers/` via joblib (the original
saves the same thing under modeling/round1 and modeling/round2_trainval),
so results can be de-scaled later.

Feature selection (2.7) first drops, for each target, any column with zero
variance on that target's X_train, from both X_train and X_test (round 1).
The `mutual_info_scores` and `xgboost_feature_importance` diagnostics are
ported verbatim from the notebook, same model hyperparameters throughout
(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
colsample_bytree=0.8, importance_type="gain"). For the final per-target
list (Solar, Wind, Hydropower, Biomass in turn), the original notebook read
`xgboost_feature_importance(..., rel_thresh=1.0)`'s printed table by eye
and hard-coded whatever came out above roughly 1% relative importance.
Those exact, real-data-derived lists have no reason to still make sense on
synthetic data's different feature/target relationships, so this script
automates the same rule instead of reusing the hard-coded lists: for each
target, the final feature set is just whatever
`xgboost_feature_importance(X_train_target, y_train_target,
importance_type="gain", rel_thresh=1.0)` returns, the same 1% threshold the
notebook used to build the table it then read by eye. The run report has
the resulting feature lists and how they compare to the original hard-coded
ones.

A note on the environment: this script imports `xgboost.XGBRegressor`,
same as the original. If xgboost isn't installed, it falls back to
`sklearn.ensemble.GradientBoostingRegressor` with whatever core
hyperparameters it can accept, using its built-in `.feature_importances_`
(impurity-based; a reasonable but not identical stand-in for XGBoost's
"gain" importance). That fallback is always reported in the output rather
than happening silently. Install xgboost and re-run for the real thing.

Input:  data/train_daily.csv, data/val_daily.csv, data/test_daily.csv
Output: data/processed/<target>_train.csv, <target>_val.csv, <target>_test.csv
        data/processed/<target>_trainval.csv, <target>_test_tv.csv
        data/processed/scalers/*.pkl
"""

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from sklearn.preprocessing import MinMaxScaler

try:
    from xgboost import XGBRegressor

    _HAS_XGBOOST = True
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor

    _HAS_XGBOOST = False

DATA_DIR = Path(__file__).resolve().parent.parent / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SCALER_DIR = PROCESSED_DIR / "scalers"

TARGETS = ["Solar", "Wind", "Hydropower", "Biomass"]
SOURCES = [t.lower() for t in TARGETS]


def load_daily_splits():
    train = pd.read_csv(DATA_DIR / "train_daily.csv", index_col=0, parse_dates=True)
    val = pd.read_csv(DATA_DIR / "val_daily.csv", index_col=0, parse_dates=True)
    test = pd.read_csv(DATA_DIR / "test_daily.csv", index_col=0, parse_dates=True)
    return train, val, test


def sqrt_input_columns(train: pd.DataFrame):
    numeric_cols = train.select_dtypes(include="number").columns.tolist()
    cyclical_mask = [c.endswith("_sin") or c.endswith("_cos") for c in numeric_cols]
    targets_mask = [c in TARGETS for c in numeric_cols]
    return [
        c
        for c, is_cyc, is_tgt in zip(numeric_cols, cyclical_mask, targets_mask)
        if (not is_cyc) and (not is_tgt)
    ]


def scale_round1(train, val, test, cols_to_sqrt):
    """Notebook cell 141 - fit MinMax on TRAIN, apply to train/val/test;
    identical copies of X per target source."""
    train_proc, val_proc, test_proc = train.copy(), val.copy(), test.copy()
    for df in (train_proc, val_proc, test_proc):
        df[cols_to_sqrt] = np.sqrt(df[cols_to_sqrt])

    X_train_all = train_proc.drop(columns=TARGETS).select_dtypes(include="number")
    X_val_all = val_proc.drop(columns=TARGETS).select_dtypes(include="number")
    X_test_all = test_proc.drop(columns=TARGETS).select_dtypes(include="number")
    assert list(X_train_all.columns) == list(X_val_all.columns) == list(X_test_all.columns)

    scaler_X = MinMaxScaler()
    X_train_scaled = pd.DataFrame(
        scaler_X.fit_transform(X_train_all), columns=X_train_all.columns, index=X_train_all.index
    )
    X_val_scaled = pd.DataFrame(
        scaler_X.transform(X_val_all), columns=X_val_all.columns, index=X_val_all.index
    )
    X_test_scaled = pd.DataFrame(
        scaler_X.transform(X_test_all), columns=X_test_all.columns, index=X_test_all.index
    )

    y = {}
    scalers_y = {}
    for target in TARGETS:
        name = target.lower()
        y_train_sqrt = np.sqrt(train_proc[[target]])
        y_val_sqrt = np.sqrt(val_proc[[target]])
        y_test_sqrt = np.sqrt(test_proc[[target]])

        scaler_y = MinMaxScaler()
        y[f"y_train_{name}"] = pd.DataFrame(
            scaler_y.fit_transform(y_train_sqrt), columns=[target], index=y_train_sqrt.index
        )
        y[f"y_val_{name}"] = pd.DataFrame(
            scaler_y.transform(y_val_sqrt), columns=[target], index=y_val_sqrt.index
        )
        y[f"y_test_{name}"] = pd.DataFrame(
            scaler_y.transform(y_test_sqrt), columns=[target], index=y_test_sqrt.index
        )
        scalers_y[name] = scaler_y

    X = {}
    for src in SOURCES:
        X[f"X_train_{src}"] = X_train_scaled.copy()
        X[f"X_val_{src}"] = X_val_scaled.copy()
        X[f"X_test_{src}"] = X_test_scaled.copy()

    return X, y, scaler_X, scalers_y


def scale_round2_trainval(train, val, test, cols_to_sqrt):
    """Notebook cell 142 - fit MinMax on TRAIN+VAL, apply to trainval/test."""
    trainval = pd.concat([train, val], axis=0).sort_index()
    trainval_proc, test_proc = trainval.copy(), test.copy()
    trainval_proc[cols_to_sqrt] = np.sqrt(trainval_proc[cols_to_sqrt])
    test_proc[cols_to_sqrt] = np.sqrt(test_proc[cols_to_sqrt])

    X_trainval_all = trainval_proc.drop(columns=TARGETS).select_dtypes(include="number")
    X_test_all = test_proc.drop(columns=TARGETS).select_dtypes(include="number")
    assert list(X_trainval_all.columns) == list(X_test_all.columns)

    scaler_X_tv = MinMaxScaler()
    X_trainval_scaled = pd.DataFrame(
        scaler_X_tv.fit_transform(X_trainval_all),
        columns=X_trainval_all.columns,
        index=X_trainval_all.index,
    )
    X_test_scaled = pd.DataFrame(
        scaler_X_tv.transform(X_test_all), columns=X_test_all.columns, index=X_test_all.index
    )

    y = {}
    scalers_y_tv = {}
    for target in TARGETS:
        name = target.lower()
        y_trainval_sqrt = np.sqrt(trainval_proc[[target]])
        y_test_sqrt = np.sqrt(test_proc[[target]])

        scaler_y_tv = MinMaxScaler()
        y[f"y_trainval_{name}"] = pd.DataFrame(
            scaler_y_tv.fit_transform(y_trainval_sqrt), columns=[target], index=y_trainval_sqrt.index
        )
        y[f"y_test_{name}_tv"] = pd.DataFrame(
            scaler_y_tv.transform(y_test_sqrt), columns=[target], index=y_test_sqrt.index
        )
        scalers_y_tv[name] = scaler_y_tv

    X = {}
    for src in SOURCES:
        X[f"X_trainval_{src}"] = X_trainval_scaled.copy()
        X[f"X_test_{src}_tv"] = X_test_scaled.copy()

    return X, y, scaler_X_tv, scalers_y_tv


def drop_zero_variance_all_x(X, targets=TARGETS):
    """Notebook 2.7.1 - verbatim port of `drop_zero_variance_all_x`."""
    dropped = {}
    for target in targets:
        name = target.lower()
        X_train = X[f"X_train_{name}"]
        X_test = X[f"X_test_{name}"]

        numeric_X_train = X_train.select_dtypes(include="number")
        selector = VarianceThreshold(threshold=0.0)
        selector.fit(numeric_X_train)
        kept_columns = numeric_X_train.columns[selector.get_support()]

        X[f"X_train_{name}"] = X_train[kept_columns]
        X[f"X_test_{name}"] = X_test[kept_columns]
        dropped[target] = X_train.shape[1] - len(kept_columns)
    return dropped


def mutual_info_scores(X_train, y_train, top_k=None, threshold=None):
    """Notebook 2.7 helper - verbatim port."""
    X_num = X_train.select_dtypes(include=["number"]).copy()

    if isinstance(y_train, pd.DataFrame):
        y = y_train.iloc[:, 0]
    elif isinstance(y_train, pd.Series):
        y = y_train
    else:
        y = np.asarray(y_train).reshape(-1)

    if isinstance(X_num, pd.DataFrame) and isinstance(y, pd.Series):
        X_num, y = X_num.align(y, join="inner", axis=0)

    valid = ~(X_num.isna().any(axis=1) | pd.isna(y))
    X_num, y = X_num.loc[valid], y.loc[valid]

    discrete_mask = (X_num.nunique() <= 10).values
    mi = mutual_info_regression(X_num, y, discrete_features=discrete_mask, random_state=0)

    scores = (
        pd.DataFrame({"features": X_num.columns, "Mutual Info Score": mi})
        .sort_values("Mutual Info Score", ascending=False)
        .reset_index(drop=True)
    )
    if threshold is not None:
        scores = scores[scores["Mutual Info Score"] > threshold].reset_index(drop=True)
    if top_k is not None:
        scores = scores.head(top_k)
    return scores


def xgboost_feature_importance(X_train, y_train, importance_type="gain", rel_thresh=0.0):
    """Notebook 2.7 helper - verbatim port (xgboost.XGBRegressor), with a
    GradientBoostingRegressor fallback, clearly flagged, if xgboost is not
    installed in this environment (see module docstring)."""
    X_num = X_train.select_dtypes(include=["number", "bool"]).copy()

    if isinstance(y_train, pd.DataFrame):
        y = y_train.iloc[:, 0].to_numpy().ravel()
    elif isinstance(y_train, pd.Series):
        y = y_train.to_numpy().ravel()
    else:
        y = np.asarray(y_train).ravel()

    if _HAS_XGBOOST:
        model = XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=-1,
            random_state=42,
        )
        model.fit(X_num, y)
        booster = model.get_booster()
        raw_imp = booster.get_score(importance_type=importance_type)
        name_map = {f"f{i}": col for i, col in enumerate(X_num.columns)}
        raw_imp = {name_map.get(k, k): v for k, v in raw_imp.items()}
        imp_series = pd.Series(raw_imp, dtype="float64").reindex(X_num.columns, fill_value=0.0)
    else:
        model = GradientBoostingRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            random_state=42,
        )
        model.fit(X_num, y)
        imp_series = pd.Series(model.feature_importances_, index=X_num.columns, dtype="float64")

    df = pd.DataFrame({"features": imp_series.index, "importance": imp_series.values})
    total = df["importance"].sum()
    df["relative_importance (%)"] = 0.0 if total == 0 else 100.0 * df["importance"] / total

    if rel_thresh > 0:
        df = df[df["relative_importance (%)"] > rel_thresh]
    return df.sort_values("relative_importance (%)", ascending=False).reset_index(drop=True)


def select_features_per_target(X, y):
    """2.7.2-2.7.5 - automated equivalent of the original notebook's
    by-eye feature pick (see module docstring)."""
    selected = {}
    importance_tables = {}
    for target in TARGETS:
        name = target.lower()
        X_train = X[f"X_train_{name}"]
        y_train = y[f"y_train_{name}"]

        imp_df = xgboost_feature_importance(
            X_train, y_train, importance_type="gain", rel_thresh=1.0
        )
        importance_tables[target] = imp_df
        features = imp_df["features"].tolist()
        selected[target] = features

        for key in (f"X_train_{name}", f"X_val_{name}", f"X_test_{name}"):
            X[key] = X[key][features]
        for key in (f"X_trainval_{name}", f"X_test_{name}_tv"):
            X[key] = X[key][features]
    return selected, importance_tables


def main():
    stats = {}
    train, val, test = load_daily_splits()
    cols_to_sqrt = sqrt_input_columns(train)
    stats["n_sqrt_cols"] = len(cols_to_sqrt)
    stats["xgboost_available"] = _HAS_XGBOOST

    X1, y1, scaler_X1, scalers_y1 = scale_round1(train, val, test, cols_to_sqrt)
    X2, y2, scaler_X2, scalers_y2 = scale_round2_trainval(train, val, test, cols_to_sqrt)

    X = {**X1, **X2}
    y = {**y1, **y2}

    dropped_zero_var = drop_zero_variance_all_x(X)
    stats["zero_variance_cols_dropped"] = dropped_zero_var

    # keep round-2 (trainval/test) X aligned to the same zero-variance
    # columns kept for round 1, per target
    for target in TARGETS:
        name = target.lower()
        kept_cols = X[f"X_train_{name}"].columns
        X[f"X_trainval_{name}"] = X[f"X_trainval_{name}"][kept_cols]
        X[f"X_test_{name}_tv"] = X[f"X_test_{name}_tv"][kept_cols]

    selected, importance_tables = select_features_per_target(X, y)
    stats["selected_features"] = selected
    stats["n_selected_features"] = {t: len(f) for t, f in selected.items()}

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    SCALER_DIR.mkdir(parents=True, exist_ok=True)

    joblib.dump(cols_to_sqrt, SCALER_DIR / "sqrt_input_columns.pkl")
    joblib.dump(scaler_X1, SCALER_DIR / "scaler_X_round1.pkl")
    joblib.dump(scaler_X2, SCALER_DIR / "scaler_X_round2_trainval.pkl")
    for name, sc in scalers_y1.items():
        joblib.dump(sc, SCALER_DIR / f"scaler_y_{name}_round1.pkl")
    for name, sc in scalers_y2.items():
        joblib.dump(sc, SCALER_DIR / f"scaler_y_{name}_round2_trainval.pkl")

    shapes = {}
    for target in TARGETS:
        name = target.lower()
        X_train = X[f"X_train_{name}"]
        X_val = X[f"X_val_{name}"]
        X_test = X[f"X_test_{name}"]
        X_trainval = X[f"X_trainval_{name}"]
        X_test_tv = X[f"X_test_{name}_tv"]

        # X and y get combined by position here, not by an index-based
        # join or concat. A handful of calendar dates sit right on the
        # boundary between two splits (the day training ends and
        # validation begins, say), and because 03_feature_engineering.py's
        # daily resample plus lag/rolling step is reproduced faithfully
        # from the notebook, that date can end up appearing twice in a
        # split's index, once from each adjacent split's partial-day
        # aggregation (see that script's docstring). X and y come from the
        # same source frame in the same row order, so they're already
        # aligned position-for-position; joining by index instead would
        # match the duplicated date label many-to-many and quietly inflate
        # the row count.
        def _attach_target(X_df, y_df, target_name):
            out = X_df.copy()
            out[target_name] = y_df[target_name].to_numpy()
            return out

        train_out = _attach_target(X_train, y[f"y_train_{name}"], target)
        val_out = _attach_target(X_val, y[f"y_val_{name}"], target)
        test_out = _attach_target(X_test, y[f"y_test_{name}"], target)
        trainval_out = _attach_target(X_trainval, y[f"y_trainval_{name}"], target)
        test_tv_out = _attach_target(X_test_tv, y[f"y_test_{name}_tv"], target)

        train_out.to_csv(PROCESSED_DIR / f"{name}_train.csv")
        val_out.to_csv(PROCESSED_DIR / f"{name}_val.csv")
        test_out.to_csv(PROCESSED_DIR / f"{name}_test.csv")
        trainval_out.to_csv(PROCESSED_DIR / f"{name}_trainval.csv")
        test_tv_out.to_csv(PROCESSED_DIR / f"{name}_test_tv.csv")

        importance_tables[target].to_csv(PROCESSED_DIR / f"{name}_xgb_importance.csv", index=False)

        shapes[target] = {
            "train": train_out.shape,
            "val": val_out.shape,
            "test": test_out.shape,
            "trainval": trainval_out.shape,
            "test_tv": test_tv_out.shape,
        }
    stats["final_shapes"] = shapes

    print("=== 04_scale_and_select.py ===")
    for k, v in stats.items():
        print(f"{k}: {v}")

    return X, y, stats


if __name__ == "__main__":
    main()


In [ ]:
%%writefile synthetic_data_repo/pipeline/requirements.txt
# Dependencies for the synthetic-data preprocessing pipeline
# (01_merge_data.py -> 02_preprocess.py -> 03_feature_engineering.py -> 04_scale_and_select.py)
pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
xgboost>=2.0
joblib>=1.3


## 4. Install the pinned requirements (keeps versions consistent with the repo)


In [ ]:
!pip install -q -r synthetic_data_repo/pipeline/requirements.txt

## 5. Run the pipeline, steps 1-4


In [ ]:
!python synthetic_data_repo/pipeline/01_merge_data.py

In [ ]:
!python synthetic_data_repo/pipeline/02_preprocess.py

In [ ]:
!python synthetic_data_repo/pipeline/03_feature_engineering.py

In [ ]:
!python synthetic_data_repo/pipeline/04_scale_and_select.py

## 6. Sanity check: confirm this run actually used real XGBoost

Check the output of the cell above (step 4) says it used `xgboost`, not the sklearn fallback. This cell just re-confirms the import succeeded.


In [ ]:
import xgboost
print('Confirmed: xgboost', xgboost.__version__, 'was available during this run.')


## 7. Save results to Google Drive

This mounts your Drive (you'll be asked to sign in or grant access the first time) and copies the final results into a folder named `synthetic_data_repo_results` in My Drive, so they can be picked up from there directly, no manual download needed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import shutil, os

dest = '/content/drive/MyDrive/synthetic_data_repo_results'
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree('synthetic_data_repo/data/processed', dest)
print('Copied results to Google Drive:', dest)
print(sorted(os.listdir(dest)))


## 8. (Backup) Zip the results and download locally

Optional: only needed if the Drive copy above didn't work for some reason.


In [ ]:
import shutil
from google.colab import files as colab_files

shutil.make_archive('processed_results', 'zip', 'synthetic_data_repo/data/processed')
print('Zipped synthetic_data_repo/data/processed -> processed_results.zip')
colab_files.download('processed_results.zip')
